# Notebook 04 — Fine-tuning LLaMA 3.1 8B avec Unsloth + LoRA

**Objectif** : Fine-tuner `unsloth/Meta-Llama-3.1-8B-bnb-4bit` sur le corpus Q&R avec LoRA (rank 32, 5 époques), anti-répétition à l’inférence ; option **contexte gold** (`USE_CONTEXT_IN_TRAINING`) pour se rapprocher d'un format avec **contexte documentaire** (gold).

> ⚠️ **GPU OBLIGATOIRE** — Aller dans `Exécution > Modifier le type d'exécution > GPU (T4 ou A100)`

**Modèle** : LLaMA 3.1 8B — même modèle que Baseline et RAG (03) → comparaison équitable des 4 méthodes.

**Inputs** :
- `BASE_PATH/data/processed/train.json`
- `BASE_PATH/data/processed/test.json`

**Outputs** :
- `BASE_PATH/models/lora_adapter/` — adaptateur LoRA
- `BASE_PATH/results/finetuned_predictions.json`

## 0. Vérification GPU

In [ ]:
# Vérification que le GPU est bien disponible avant de continuer
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Aucun GPU détecté !\n"
        "→ Allez dans Exécution > Modifier le type d'exécution > GPU, puis relancez."
    )

gpu_name   = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU détecté  : {gpu_name}")
print(f"VRAM totale  : {gpu_mem_gb:.1f} Go")
print(f"CUDA version : {torch.version.cuda}")

## 1. Montage Google Drive

In [ ]:
# Montage du Drive et définition du chemin de base du projet
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/llm-integration-study/'

## 2. Installation des dépendances

In [ ]:
# Installation robuste Unsloth + Unsloth Zoo (versions cohérentes)
# IMPORTANT: exécuter cette cellule, puis redémarrer le runtime avant de continuer.
!pip uninstall -y unsloth unsloth_zoo trl transformers peft accelerate bitsandbytes xformers
!pip install --no-cache-dir -U git+https://github.com/unslothai/unsloth-zoo.git
!pip install --no-cache-dir -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
print("Installation Unsloth terminée. Redémarre maintenant le runtime (Runtime > Restart runtime).")

In [ ]:
# Dépendances complémentaires (hors stack déjà gérée par Unsloth)
!pip install -q datasets trl rouge-score bert-score

## 3. Imports et configuration

In [ ]:
# Imports de toutes les bibliothèques nécessaires
import os
import json
import time
import re
import string
import numpy as np
import torch
from tqdm.notebook import tqdm
from datasets import Dataset

# Unsloth DOIT être importé avant trl/transformers/peft
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

# Chemins Drive
PROCESSED_PATH = os.path.join(BASE_PATH, 'data', 'processed')
MODELS_PATH    = os.path.join(BASE_PATH, 'models', 'lora_adapter_ft')
RESULTS_PATH   = os.path.join(BASE_PATH, 'results')

for path in [MODELS_PATH, RESULTS_PATH]:
    os.makedirs(path, exist_ok=True)

# LLaMA 3.1 8B — même modèle que Baseline/RAG → comparaison équitable
BASE_MODEL   = "unsloth/Meta-Llama-3.1-8B-bnb-4bit"
LORA_RANK    = 32
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
TARGET_MODS  = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Profil auto selon GPU (priorité A100/H100)
gpu_name = torch.cuda.get_device_name(0).lower() if torch.cuda.is_available() else "cpu"
if "a100" in gpu_name or "h100" in gpu_name:
    MAX_SEQ_LEN = 1024
    BATCH_SIZE = 2
    GRAD_ACC_STEPS = 8
    MAX_CONTEXT_CHARS = 1200
    GPU_PROFILE = "A100/H100"
elif "l4" in gpu_name:
    MAX_SEQ_LEN = 896
    BATCH_SIZE = 1
    GRAD_ACC_STEPS = 16
    MAX_CONTEXT_CHARS = 900
    GPU_PROFILE = "L4"
else:
    MAX_SEQ_LEN = 768
    BATCH_SIZE = 1
    GRAD_ACC_STEPS = 16
    MAX_CONTEXT_CHARS = 700
    GPU_PROFILE = "T4-safe"

NUM_EPOCHS   = 6
LR           = 8e-5
VAL_RATIO    = 0.10
SEED         = 42

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print("Configuration :")
print(f"  GPU profile      : {GPU_PROFILE} ({gpu_name})")
print(f"  Modèle de base   : {BASE_MODEL}")
print(f"  LoRA rank/alpha  : {LORA_RANK}/{LORA_ALPHA}")
print(f"  max_seq_length   : {MAX_SEQ_LEN}")
print(f"  batch/grad_acc   : {BATCH_SIZE}/{GRAD_ACC_STEPS}")
print(f"  max context char : {MAX_CONTEXT_CHARS}")
print(f"  Époques          : {NUM_EPOCHS}")
print(f"  Learning rate    : {LR}")
print(f"  Validation ratio : {VAL_RATIO}")
print(f"  LoRA output path : {MODELS_PATH}")

## 4. Chargement des données

In [ ]:
# Chargement de train.json et test.json depuis Drive
def load_json(path):
    """Charge un fichier JSON avec gestion d'erreur."""
    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"  Chargé : {path} ({len(data)} entrées)")
        return data
    except FileNotFoundError:
        print(f"  [ERROR] Fichier introuvable : {path}")
        print("  → Assurez-vous d'avoir exécuté 02 ou 02b (dataset) au préalable.")
        return []
    except json.JSONDecodeError as e:
        print(f"  [ERROR] JSON invalide : {e}")
        return []

print("Chargement des datasets...")
train_data = load_json(os.path.join(PROCESSED_PATH, 'train.json'))
test_data  = load_json(os.path.join(PROCESSED_PATH, 'test.json'))

print(f"\nTrain : {len(train_data)} paires")
print(f"Test  : {len(test_data)} questions")

## 5. Formatage des exemples SFT (Alpaca ± contexte)

In [ ]:
# Formatage pour le SFTTrainer
ALPACA_TEMPLATE = (
    "### Instruction: Réponds à cette question en français, de façon concise et fidèle.\n"
    "### Input: {question}\n"
    "### Response: {answer}"
)

# Pour améliorer le fine-tuning seul, on active le contexte gold en entraînement.
# L'inférence 04 reste sans retrieval externe (pas de FAISS), donc méthode "fine-tunée seule".
USE_CONTEXT_IN_TRAINING = True
# MAX_CONTEXT_CHARS est défini automatiquement selon le GPU dans la cellule de configuration.

CONTEXT_TEMPLATE = (
    "### Instruction: Réponds à cette question en te basant sur le contexte ci-dessous.\n"
    "### Context: {context}\n"
    "### Input: {question}\n"
    "### Response: {answer}"
)

def format_training_example(item):
    q = item.get('question', '').strip()
    a = item.get('answer', '').strip()
    ctx = str(item.get('context', '')).strip()
    if USE_CONTEXT_IN_TRAINING and len(ctx) >= 40:
        ctx = ctx[:MAX_CONTEXT_CHARS]
        text = CONTEXT_TEMPLATE.format(context=ctx, question=q, answer=a)
    else:
        text = ALPACA_TEMPLATE.format(question=q, answer=a)
    return {"text": text}

formatted_train = [format_training_example(item) for item in train_data if item.get('question') and item.get('answer')]
hf_dataset = Dataset.from_list(formatted_train)

split = hf_dataset.train_test_split(test_size=VAL_RATIO, seed=SEED, shuffle=True)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Dataset SFT prêt : total={len(hf_dataset)} train={len(train_dataset)} val={len(eval_dataset)}")
print(f"  USE_CONTEXT_IN_TRAINING = {USE_CONTEXT_IN_TRAINING}")
print("\nAperçu du premier exemple :")
print(train_dataset[0]['text'][:500])

## 6. Chargement du modèle et application de LoRA

In [ ]:
# Chargement de LLaMA 3.1 8B quantifié en 4-bit via Unsloth
print(f"Chargement de {BASE_MODEL}...")
print("(Téléchargement ~4 Go depuis HuggingFace, peut prendre 5-10 min)")

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,          # Auto-détection (bfloat16 sur A100, float16 sur T4)
        load_in_4bit=True,   # Quantification 4-bit pour tenir en VRAM
    )
    print("Modèle de base chargé.")
except Exception as e:
    raise RuntimeError(f"Échec du chargement du modèle : {e}")

In [ ]:
# Application de l'adaptateur LoRA sur les couches cibles
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=TARGET_MODS,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Économie mémoire supplémentaire
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

# Comptage des paramètres entraînables
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Paramètres entraînables : {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 7. Entraînement

In [ ]:
# Configuration du SFTTrainer avec validation (checkpoint meilleur modèle)
training_args = TrainingArguments(
    output_dir="/content/tmp_checkpoints_ft",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACC_STEPS,
    warmup_steps=30,
    max_grad_norm=1.0,
    learning_rate=LR,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

print("Trainer configuré. Démarrage de l'entraînement...")

In [ ]:
# Lancement de l'entrainement (profil memoire T4)
if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_start = time.time()

try:
    trainer_stats = trainer.train()
    train_duration = time.time() - train_start
    print(f"\nEntrainement termine en {train_duration/60:.1f} minutes")
    print(f"Loss finale      : {trainer_stats.training_loss:.4f}")
    print(f"Steps total      : {trainer_stats.global_step}")
except Exception as e:
    raise RuntimeError(f"Echec de l'entrainement : {e}")

## 8. Sauvegarde de l'adaptateur LoRA sur Drive

In [ ]:
# Sauvegarde de l'adaptateur LoRA (uniquement les poids delta, ~50 Mo)
try:
    model.save_pretrained(MODELS_PATH)
    tokenizer.save_pretrained(MODELS_PATH)
    print(f"Adaptateur LoRA sauvegardé : {MODELS_PATH}")

    # Listage des fichiers produits
    files = os.listdir(MODELS_PATH)
    for fname in files:
        fpath = os.path.join(MODELS_PATH, fname)
        size  = os.path.getsize(fpath) / 1024
        print(f"  {fname:<40} {size:>8.1f} Ko")
except Exception as e:
    print(f"[ERROR] Sauvegarde LoRA : {e}")

## 9. Inférence sur le test set

In [ ]:
# Passage en mode inférence
FastLanguageModel.for_inference(model)

_ALPACA_STOP_MARKERS = ["\n### Instruction:", "\n### Input:", "\n### Response:", "\n### Context:"]
_NOISE_PATTERNS = [
    r"pour sauvegarder cet article.*$",
    r"connectez-vous.*$",
    r"abonnez-vous.*$",
    r"cookies?.*$",
]

def _cleanup_answer(text):
    out = (text or "").strip()
    for marker in _ALPACA_STOP_MARKERS:
        if marker in out:
            out = out.split(marker)[0].strip()
    for p in _NOISE_PATTERNS:
        out = re.sub(p, "", out, flags=re.IGNORECASE | re.MULTILINE).strip()
    out = re.sub(r"(\b.{1,40}?\b)(\s+\1){2,}", r"\1", out, flags=re.IGNORECASE)
    if len(out) > 40 and out[-1] not in ".!?":
        cut = max(out.rfind("."), out.rfind("!"), out.rfind("?"))
        if cut > 40:
            out = out[:cut+1]
    return out.strip()

def _confidence_from_scores(gen_outputs):
    # score proxy: moyenne des probas max de chaque token généré
    if not getattr(gen_outputs, "scores", None):
        return None
    probs = [torch.softmax(s[0], dim=-1).max().item() for s in gen_outputs.scores]
    if not probs:
        return None
    return round(float(np.mean(probs)), 4)

def generate_answer(question, max_new_tokens=360):
    prompt = (
        "### Instruction: Réponds à cette question en français, de façon concise et fidèle.\n"
        f"### Input: {question}\n"
        "### Response:"
    )
    try:
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        start = time.time()
        with torch.no_grad():
            gen = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                min_new_tokens=20,
                do_sample=False,
                repetition_penalty=1.16,
                no_repeat_ngram_size=4,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id,
                return_dict_in_generate=True,
                output_scores=True,
            )
        latency_ms = round((time.time() - start) * 1000)
        prompt_len = inputs["input_ids"].shape[1]
        new_tokens = gen.sequences[0][prompt_len:]
        answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        answer = _cleanup_answer(answer)
        confidence = _confidence_from_scores(gen)
        truncated = len(new_tokens) >= max_new_tokens
        return answer, latency_ms, confidence, truncated
    except Exception as e:
        print(f"  [ERROR] Génération : {e}")
        return "", 0, None, False

print("Modèle en mode inférence. Lancement sur le test set...")

In [ ]:
# Inférence sur tout le test set avec barre de progression
finetuned_predictions = []

for item in tqdm(test_data, desc="Inférence fine-tuné"):
    question = item.get('question', '')
    true_answer = item.get('answer', '')

    predicted, latency, confidence, truncated = generate_answer(question)

    finetuned_predictions.append({
        "pair_id": item.get('pair_id', ''),
        "question": question,
        "predicted_answer": predicted,
        "true_answer": true_answer,
        "latency_ms": latency,
        "confidence": confidence,
        "truncated": truncated,
        "method": "finetuned"
    })

latencies = [p['latency_ms'] for p in finetuned_predictions if p['latency_ms'] > 0]
trunc_rate = 100 * np.mean([p.get("truncated", False) for p in finetuned_predictions]) if finetuned_predictions else 0.0
conf_vals = [p["confidence"] for p in finetuned_predictions if p.get("confidence") is not None]
print(f"\nInférence terminée : {len(finetuned_predictions)} prédictions")
print(f"Latence moyenne    : {np.mean(latencies):.0f} ms" if latencies else "Latence : N/A")
print(f"Taux troncature    : {trunc_rate:.1f}%")
print(f"Confiance moyenne  : {np.mean(conf_vals):.3f}" if conf_vals else "Confiance : N/A")

In [ ]:
# Sauvegarde des prédictions du modèle fine-tuné sur Drive
finetuned_path = os.path.join(RESULTS_PATH, 'finetuned_predictions.json')
try:
    with open(finetuned_path, 'w', encoding='utf-8') as f:
        json.dump(finetuned_predictions, f, ensure_ascii=False, indent=2)
    print(f"Prédictions sauvegardées : {finetuned_path}")
    print(f"  → {len(finetuned_predictions)} entrées, {os.path.getsize(finetuned_path)/1024:.1f} Ko")
except Exception as e:
    print(f"[ERROR] Sauvegarde prédictions : {e}")

In [ ]:
# Mini-évaluation immédiate (utile quand on exécute les notebooks séparément)
_rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def _norm(t):
    t = (t or "").lower().strip()
    t = t.translate(str.maketrans('', '', string.punctuation))
    return " ".join(t.split())

def _f1(p, g):
    pt, gt = _norm(p).split(), _norm(g).split()
    if not pt or not gt:
        return 0.0
    inter = len(set(pt) & set(gt))
    if inter == 0:
        return 0.0
    pr, rc = inter / len(pt), inter / len(gt)
    return 2 * pr * rc / (pr + rc)

preds = [x.get("predicted_answer","") for x in finetuned_predictions]
refs  = [x.get("true_answer","") for x in finetuned_predictions]
em = np.mean([int(_norm(p) == _norm(g)) for p, g in zip(preds, refs)]) * 100
f1 = np.mean([_f1(p, g) for p, g in zip(preds, refs)]) * 100
rl = np.mean([_rouge.score(g if g else " ", p if p else " ")["rougeL"].fmeasure for p, g in zip(preds, refs)]) * 100
try:
    _, _, F = bert_score_fn(preds if preds else [" "], refs if refs else [" "], lang="fr",
                            model_type="distilbert-base-multilingual-cased", batch_size=32, verbose=False)
    bs = float(F.mean()) * 100
except Exception as e:
    print(f"[WARN] BERTScore indisponible: {e}")
    bs = 0.0
print("\n--- Mini-évaluation Fine-tuné (NB04) ---")
print(f"EM={em:.1f}% | F1={f1:.1f}% | ROUGE-L={rl:.1f}% | BERTScore={bs:.1f}%")
# Mini-évaluation immédiate (utile quand on exécute les notebooks séparément)
_rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def _norm(t):
    t = (t or "").lower().strip()
    t = t.translate(str.maketrans('', '', string.punctuation))
    return " ".join(t.split())

def _f1(p, g):
    pt, gt = _norm(p).split(), _norm(g).split()
    if not pt or not gt:
        return 0.0
    inter = len(set(pt) & set(gt))
    if inter == 0:
        return 0.0
    pr, rc = inter / len(pt), inter / len(gt)
    return 2 * pr * rc / (pr + rc)

preds = [x.get("predicted_answer","") for x in finetuned_predictions]
refs  = [x.get("true_answer","") for x in finetuned_predictions]
em = np.mean([int(_norm(p) == _norm(g)) for p, g in zip(preds, refs)]) * 100
f1 = np.mean([_f1(p, g) for p, g in zip(preds, refs)]) * 100
rl = np.mean([_rouge.score(g if g else " ", p if p else " ")["rougeL"].fmeasure for p, g in zip(preds, refs)]) * 100
try:
    _, _, F = bert_score_fn(preds if preds else [" "], refs if refs else [" "], lang="fr",
                            model_type="distilbert-base-multilingual-cased", batch_size=32, verbose=False)
    bs = float(F.mean()) * 100
except Exception as e:
    print(f"[WARN] BERTScore indisponible: {e}")
    bs = 0.0
print("\n--- Mini-évaluation Fine-tuné (NB04) ---")
print(f"EM={em:.1f}% | F1={f1:.1f}% | ROUGE-L={rl:.1f}% | BERTScore={bs:.1f}%")
print(f"Résultat sauvegardé: {finetuned_path}")

## 10. Résumé final

In [ ]:
# Affichage complet du résumé de ce qui a été produit
latencies = [p['latency_ms'] for p in finetuned_predictions if p['latency_ms'] > 0]
conf_vals = [p["confidence"] for p in finetuned_predictions if p.get("confidence") is not None]

print("=" * 65)
print("RÉSUMÉ — Notebook 04 : Fine-tuning")
print("=" * 65)
print(f"\nModèle de base      : {BASE_MODEL}")
print(f"LoRA rank / alpha   : {LORA_RANK} / {LORA_ALPHA}")
print(f"Modules cibles      : {TARGET_MODS}")
print(f"Époques             : {NUM_EPOCHS}")
print(f"Exemples entraînem. : train={len(train_dataset)} / val={len(eval_dataset)}")
try:
    print(f"Loss finale         : {trainer_stats.training_loss:.4f}")
    print(f"Durée entraînem.    : {train_duration/60:.1f} min")
except Exception:
    pass
print(f"Prédictions test    : {len(finetuned_predictions)}")
print(f"Latence moy. infér. : {np.mean(latencies):.0f} ms" if latencies else "Latence : N/A")
print(f"Confiance moyenne   : {np.mean(conf_vals):.3f}" if conf_vals else "Confiance : N/A")

print(f"\nFichiers produits :")
for fpath in [finetuned_path]:
    try:
        print(f"  {fpath}  ({os.path.getsize(fpath)/1024:.1f} Ko)")
    except Exception:
        print(f"  {fpath}")
print(f"  {MODELS_PATH}/  (adaptateur LoRA)")

print("\n✔ Notebook 04 terminé. Lancez `05_raft.ipynb`, puis `06_rag_rerank.ipynb`, `07_function_calling.ipynb`, `08_ft_plus_rag.ipynb`, puis `09_evaluation.ipynb`.")
print("=" * 65)